22h05m27 to 05h59m57 (8 часов записи)

In [ ]:
from collections import defaultdict
import re

filelist_path = "untracked/files.txt"

counts = {
        's_mp4': 0,
        'yolo_counts': defaultdict(int),  # counts for each yolo[i]
    }
    
# Regular expression to match yolo pattern
# Matches patterns like .yolo0.mp4, .yolo1.mp4, etc.
yolo_pattern = re.compile(r'\.yolo(\d+)\.mp4$')


with open(filelist_path, 'r') as f:
    for line in f:
        filename = line.strip()
        if not filename:  # Skip empty lines
            continue
        
        # Check if it's an MP4 file
        if filename.endswith('s.mp4'):
            counts['s_mp4'] += 1
            
        yolo_match = yolo_pattern.search(filename)
        if yolo_match:
            yolo_index = int(yolo_match.group(1))
            counts['yolo_counts'][yolo_index] += 1

counts

Из 16ти
- 13 произошли в диапазоне от 3 утра до 3:50 утра (на боку) - тот же человек дважды
ошибочные:
- vzuev_june18_2025__18-JUN-2025_02h14m21.683s.yolo1 // 0 также ничего; человека нет на видео
- vzuev_june18_2025__18-JUN-2025_04h55m29.583s.yolo1
- vzuev_june18_2025__18-JUN-2025_05h00m29.647s.yolo1 // 0 также ничего не определил; человека на видео вообще нет в этот момент


# Метрики тренировки

обработка результатов работы скрипта `eval.py`

In [ ]:
import json

with open('untracked/arch_comparison/metrics.json') as f:
    metrics = json.load(f)

list(metrics.keys())

In [ ]:
import pandas as pd

rows = []

for architecture, experiments in metrics.items():
    for experiment in experiments:
        m1 = experiment["metrics"]
        m2 = experiment["metrics_subj2"]

        rows.append({
            "architecture": architecture,

            "subj1_acc":  m1["accuracy"],
            "subj1_prec": m1["precision"],
            "subj1_rec":  m1["recall"],
            "subj1_f1":   m1["f1_score"],

            "subj2_acc":  m2["accuracy"],
            "subj2_prec": m2["precision"],
            "subj2_rec":  m2["recall"],
            "subj2_f1":   m2["f1_score"],
        })

df = pd.DataFrame(rows)
df

In [ ]:
metrics = [
    "subj1_acc", "subj1_prec", "subj1_rec", "subj1_f1",
    "subj2_acc", "subj2_prec", "subj2_rec", "subj2_f1",
]

summary = (
    df.groupby("architecture")[metrics]
      .agg(["mean", "std"])
)
summary.round(1)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df_long = df.melt(
    id_vars="architecture",
    value_vars=metrics,
    var_name="metric",
    value_name="value",
)

g = sns.catplot(
    data=df_long,
    x="architecture",
    y="value",
    col="metric",
    kind="violin",
    inner="box",
    cut=0,
    col_wrap=4,
    height=4,
    aspect=1.3,
    sharey=False,
)

g.set_xticklabels(rotation=45, ha="right")
g.set_axis_labels("", "Score (%)")
g.set_titles("{col_name}")

plt.tight_layout()
plt.show()